In [1]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import pickle
import pandas as pd


from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk


In [ ]:
df = pd.read_csv('SpamMessages.csv')
X = df.drop('Category', axis=1)

nltk.download('punkt_tab')
nltk.download('stopwords')

#lowercase
X['Message'] = X['Message'].str.lower()

#removes punctuation
X['Message'] = X['Message'].str.replace(r'[^\w\s]', '', regex=True)

#splits each word
X['Tokens'] = X['Message'].apply(word_tokenize)

#downloads stop words
stopWords = stopwords.words('english')

#removes stop words from the list of tokens like "the", "is", "in", etc.
X['Tokens'] = X['Tokens'].apply(lambda words: [w for w in words if w not in stopWords])

#joins all the words together
X['Final Message'] = X['Tokens'].apply(lambda x: ' '.join(x))


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
#turns the final message into a vector of numbers
tfidf = TfidfVectorizer()
X_final = tfidf.fit_transform(X['Final Message'])
print(X_final.shape)

(5572, 9411)


In [5]:
y = df['Category']
y = y.map({'ham': 0, 'spam': 1})

In [ ]:
#splits the data into 75% training them model and 25% testing the model
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.25, random_state=42)

In [ ]:
#logistic regression model with balanced class weights to account for any imbalance in the dataset
model = LogisticRegression(class_weight='balanced')

#fit the model to data
model.fit(X_train, y_train)

#make predictions
preds = model.predict(X_test)
print(f"Accuracy Score: {accuracy_score(y_test, preds)}")

#classification report
print(classification_report(y_test, preds))

Accuracy Score: 0.9770279971284996
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1207
           1       0.92      0.91      0.91       186

    accuracy                           0.98      1393
   macro avg       0.95      0.95      0.95      1393
weighted avg       0.98      0.98      0.98      1393



In [9]:
with open('spam.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)
    
with open('tfidf_model.pkl', 'wb') as vec_file:
    pickle.dump(tfidf, vec_file)